Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computacion  
Inteligencia Artificial - Seccion 10

Nadissa Vela - 23764  
Cristian Tunchez - 231359

---

# Laboratorio No. 6

## Task 2 - Implementacion Connect Four (4 en linea)

En este laboratorio implementamos una IA para **Connect Four** utilizando **Minimax** con profundidad limitada.  

La utilidad en estados terminales se define como:  

- **+1000** si gana la IA.  
- **-1000** si gana el oponente.  
- **0** si hay empate.

### Task 2.1

En esta parte se implementa:  

- La clase `Connect4` para modelar estado, acciones y terminalidad.
- El agente **Minimax recursivo** mediante `get_best_move(board, depth)`.
- Restriccion de profundidad recomendada ($ d = 3 $ o $ d = 4 $).

#### 1. Importaciones y constantes del entorno

En esta seccion se importan utilidades basicas y se definen constantes globales.

In [60]:
from copy import deepcopy
import random
import math

ROWS = 6
COLS = 7
EMPTY = " "
PLAYER_1 = "X"
PLAYER_2 = "O"

#### 2. Clase Connect4

Esta parte implementa la logica y modelo principal del entorno de juego:  

- `actions(s)`: columnas validas para jugar.  

- `make_move(col, player)`: aplica una jugada respetando gravedad.  
- `succ(s, a)`: genera estado sucesor.  

- `check_winner(player)`: detecta 4 en linea horizontal, vertical y diagonal.  

- `is_terminal(s)`: determina fin del juego por victoria o empate.

In [61]:
class Connect4:

    def __init__(self, board=None):
        if board is None:
            self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]
        else:
            self.board = deepcopy(board)

    def copy(self):
        return Connect4(self.board)

    def actions(self):
        # Columnas validas: la celda superior aun esta vacia.
        return [c for c in range(COLS) if self.board[0][c] == EMPTY]

    def make_move(self, col, player):
        if col not in self.actions():
            return False
        for r in range(ROWS - 1, -1, -1):
            if self.board[r][col] == EMPTY:
                self.board[r][col] = player
                return True
        return False

    def succ(self, action, player):
        next_state = self.copy()
        next_state.make_move(action, player)
        return next_state

    def check_winner(self, player):
        b = self.board
        # Horizontal
        for r in range(ROWS):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r][c + 1] == player and b[r][c + 2] == player and b[r][c + 3] == player:
                    return True

        # Vertical
        for r in range(ROWS - 3):
            for c in range(COLS):
                if b[r][c] == player and b[r + 1][c] == player and b[r + 2][c] == player and b[r + 3][c] == player:
                    return True

        # Diagonal descendente (\\)
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r + 1][c + 1] == player and b[r + 2][c + 2] == player and b[r + 3][c + 3] == player:
                    return True

        # Diagonal ascendente (/)
        for r in range(3, ROWS):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r - 1][c + 1] == player and b[r - 2][c + 2] == player and b[r - 3][c + 3] == player:
                    return True

        return False

    def is_terminal(self):
        return self.check_winner(PLAYER_1) or self.check_winner(PLAYER_2) or len(self.actions()) == 0

    def winner(self):
        if self.check_winner(PLAYER_1):
            return PLAYER_1

        if self.check_winner(PLAYER_2):
            return PLAYER_2

        return None

    def print_board(self):
        # Muestra fila superior primero con marco decorativo.
        line = "*" * (COLS * 4 + 5)
        print()
        print(line)
        for r in range(ROWS):
            print("* | " + " | ".join(self.board[r]) + " | *")
        print(line)
        print()

#### 3. Agente Minimax y seleccion de jugada

Aqui se implementa la función `get_best_move(board, depth)` que utiliza el algoritmo **Minimax recursivo**:  

- Nodo MAX: turno de la IA (maximiza valor).  

- Nodo MIN: turno del oponente (minimiza valor).  

- Casos base: estado terminal o profundidad 0.

In [62]:
def get_best_move(board, depth=4, ai_player=PLAYER_1):
    # La llamada inicial comienza como el jugador que maximiza (la IA).
    best_score = float("-inf")
    best_move = -1  # Inicializar con un movimiento inválido

    for col in board.actions():
        # Crear un estado sucesor realizando el movimiento para el jugador IA
        next_state = board.succ(col, ai_player)

        # Llamar al algoritmo minimax para el oponente (jugador que minimiza)
        # desde el estado sucesor, con profundidad reducida.
        score = _minimax(next_state, depth - 1, False, ai_player)

        if score > best_score:
            best_score = score
            best_move = col

    return best_move

def _minimax(state, depth, maximizing_player, ai_player):
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    # Caso base 1: Estado terminal alcanzado (Win/Loss/Draw)
    # Si el estado es terminal, la utilidad U(s) se define directamente.
    if state.is_terminal():
        winner = state.winner()
        if winner == ai_player:
            return 1000  # La IA gana: U(s) = +1000
        if winner == opponent:
            return -1000  # El oponente gana: U(s) = -1000
        return 0  # Empate: U(s) = 0

    # Caso base 2: Límite de profundidad alcanzado sin un estado terminal.
    # Según la solicitud del usuario, no se utiliza una heurística (H(s) = 0),
    # por lo que se devuelve una puntuación neutral.
    if depth == 0:
        return 0

    if maximizing_player:
        # Nodo MAX: Busca maximizar el valor V(s) = max_{a \in A(s)} V(s, a)
        max_eval = float("-inf")
        for col in state.actions():
            child = state.succ(col, ai_player)
            # Llamada recursiva: el siguiente jugador es minimizador
            eval = _minimax(child, depth - 1, False, ai_player)
            max_eval = max(max_eval, eval)
        return max_eval
    else:  # Jugador que minimiza
        # Nodo MIN: Busca minimizar el valor V(s) = min_{a \in A(s)} V(s, a)
        min_eval = float("inf")
        for col in state.actions():
            child = state.succ(col, opponent)
            # Llamada recursiva: el siguiente jugador es maximizador
            eval = _minimax(child, depth - 1, True, ai_player)
            min_eval = min(min_eval, eval)
        return min_eval

#### 4. Simulación

Esta seccion ejecuta una prueba rapida de funcionamiento:  

- IA con Minimax contra un oponente aleatorio.  

- Se imprime el tablero final y el ganador.

In [63]:
# Prueba Agente Minimax vs Agente Aleatorio
# Se puede ajustar la profundidad (depth) y cuál jugador es la IA.
def play_minimax_vs_random(depth=5, ai_player=PLAYER_1, seed=None):
    if seed is not None:
        random.seed(seed)
    game = Connect4()
    current_player = PLAYER_1

    print(f"\n[INICIO DEL JUEGO] Agente Minimax (profundidad {depth}) vs. Agente Aleatorio")
    game.print_board()

    while not game.is_terminal():
        if current_player == ai_player:
            print(f"Turno del Agente Minimax ({ai_player})... ")
            # Usamos get_best_move para que la IA juegue
            col = get_best_move(game, depth=depth, ai_player=ai_player)
            if col is None: # Si no hay movimientos válidos (ej. tablero lleno)
                break
            game.make_move(col, current_player)
            print(f"Minimax juega en columna {col + 1}")
        else:
            print(f"Turno del Agente Aleatorio ({current_player})... ")
            valid_actions = game.actions()
            if not valid_actions:
                break # No hay movimientos válidos
            col = random.choice(valid_actions)
            game.make_move(col, current_player)
            print(f"Aleatorio juega en columna {col + 1}")

        game.print_board()
        current_player = PLAYER_1 if current_player == PLAYER_2 else PLAYER_2

    print("¡Fin del Juego!")
    w = game.winner()
    if w is None:
        return "Empate"
    else:
        return f"Gana jugador {w}"


# Ejecutar la prueba
print("\n--- Demostración: Agente Minimax (X) vs. Agente Aleatorio (O) ---")
resultado_ia_vs_random = play_minimax_vs_random(depth=4, ai_player=PLAYER_1, seed=42)
print("[FIN] Resultado Final:", resultado_ia_vs_random)


--- Demostración: Agente Minimax (X) vs. Agente Aleatorio (O) ---

[INICIO DEL JUEGO] Agente Minimax (profundidad 4) vs. Agente Aleatorio

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
*********************************

Turno del Agente Minimax (X)... 
Minimax juega en columna 1

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* | X |   |   |   |   |   |   | *
*********************************

Turno del Agente Aleatorio (O)... 
Aleatorio juega en columna 6

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   

### Task 2.2 - Optimizacion con Poda Alfa-Beta

En esta parte optimizamos el agente Minimax incorporando **Poda Alfa-Beta**:  

- `alpha`: mejor valor (cota inferior) que MAX puede garantizar hasta el momento.  

- `beta`: mejor valor (cota superior) que MIN puede garantizar hasta el momento.  

- Si en algun punto `alpha >= beta`, se **poda** ese subarbol porque ya no puede cambiar la decision final.  

Ademas, para demostrar el impacto, se cuentan los nodos visitados por:  

1. Minimax puro  

2. Minimax con Alfa-Beta  

La comparacion se hace sobre **el mismo estado del tablero** y la misma profundidad de busqueda.

In [64]:
# Implementacion instrumentada para comparar nodos visitados.

def minimax_count_nodes(state, depth, maximizing_player, ai_player, counter):
    counter["nodes"] += 1
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2
    if state.is_terminal():
        winner = state.winner()

        if winner == ai_player:
            return None, 1000

        if winner == opponent:
            return None, -1000
        return None, 0

    if depth == 0:
        # Se devuelve una puntuación neutral.
        return None, 0

    valid_moves = state.actions()
    # Si no hay movimientos válidos y no es terminal, es un empate (se devuelve 0).
    if not valid_moves:
        return None, 0

    if maximizing_player:
        best_score = float("-inf")
        best_move = valid_moves[0]

        for col in valid_moves:
            child = state.succ(col, ai_player)
            _, score = minimax_count_nodes(child, depth - 1, False, ai_player, counter)

            if score > best_score:
                best_score = score
                best_move = col

        return best_move, best_score

    best_score = float("inf")
    best_move = valid_moves[0]

    for col in valid_moves:
        child = state.succ(col, opponent)
        _, score = minimax_count_nodes(child, depth - 1, True, ai_player, counter)

        if score < best_score:
            best_score = score
            best_move = col

    return best_move, best_score

def alphabeta_count_nodes(state, depth, alpha, beta, maximizing_player, ai_player, counter):
    counter["nodes"] += 1
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if state.is_terminal():
        winner = state.winner()

        if winner == ai_player:
            return None, 1000

        if winner == opponent:
            return None, -1000

        return None, 0

    if depth == 0:
        # Se devuelve una puntuación neutral.
        return None, 0

    valid_moves = state.actions()
    # Si no hay movimientos válidos y no es terminal, es un empate (se devuelve 0).
    if not valid_moves:
        return None, 0

    if maximizing_player:
        best_score = float("-inf")
        best_move = valid_moves[0]

        for col in valid_moves:
            child = state.succ(col, ai_player)
            _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, False, ai_player, counter)

            if score > best_score:
                best_score = score
                best_move = col

            # Actualiza alpha en nodo MAX.
            alpha = max(alpha, best_score)

            # Poda: MIN ya tiene una opcion mejor o igual en ancestros.
            if alpha >= beta:
                break

        return best_move, best_score

    best_score = float("inf")
    best_move = valid_moves[0]

    for col in valid_moves:
        child = state.succ(col, opponent)
        _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, True, ai_player, counter)

        if score < best_score:
            best_score = score
            best_move = col

        # Actualiza beta en nodo MIN.
        beta = min(beta, best_score)

        # Poda: MAX ya tiene una opcion mejor o igual en ancestros.
        if alpha >= beta:
            break

    return best_move, best_score

def get_best_move_alphabeta(board, depth=4, ai_player=PLAYER_1):
    state = board if isinstance(board, Connect4) else Connect4(board)
    move, _ = alphabeta_count_nodes(
        state=state,
        depth=depth,
        alpha=float("-inf"),
        beta=float("inf"),
        maximizing_player=True,
        ai_player=ai_player,
        counter={"nodes": 0},
    )

    return move

**Demostracion: Minimax vs Alfa-Beta (nodos visitados)**

Se construye un estado intermedio fijo del tablero y se ejecutan ambos algoritmos con la misma profundidad.  

Salida esperada:  

- Mejor movimiento sugerido por cada algoritmo.  

- Puntaje estimado por cada algoritmo.  

- Nodos visitados por Minimax puro.  

- Nodos visitados por Alfa-Beta.  

- Porcentaje de reduccion de nodos (deberia ser significativo).

In [65]:
def build_demo_state():
    # Secuencia fija de jugadas para crear un estado no terminal reproducible.
    # PLAYER_1 y PLAYER_2 alternan turnos.
    moves = [3, 2, 3, 2, 4, 2, 4, 1, 5, 1]
    game = Connect4()
    current = PLAYER_1

    for col in moves:
        game.make_move(col, current)
        current = PLAYER_1 if current == PLAYER_2 else PLAYER_2

    return game

depth_test = 4
ai_player = PLAYER_1
state_demo = build_demo_state()
print("Tablero de prueba (estado compartido):")
state_demo.print_board()

# Minimax puro con contador.
counter_minimax = {"nodes": 0}
move_mm, score_mm = minimax_count_nodes(
    state=state_demo,
    depth=depth_test,
    maximizing_player=True,
    ai_player=ai_player,
    counter=counter_minimax,
)

# Alfa-Beta con contador.
counter_ab = {"nodes": 0}
move_ab, score_ab = alphabeta_count_nodes(
    state=state_demo,
    depth=depth_test,
    alpha=float("-inf"),
    beta=float("inf"),
    maximizing_player=True,
    ai_player=ai_player,
    counter=counter_ab,
)

nodes_mm = counter_minimax["nodes"]
nodes_ab = counter_ab["nodes"]
reduction = 0.0 if nodes_mm == 0 else (1 - (nodes_ab / nodes_mm)) * 100
print(f"Minimax puro -> mejor mov: {move_mm}, score: {score_mm}, nodos: {nodes_mm}")
print(f"Alfa-Beta    -> mejor mov: {move_ab}, score: {score_ab}, nodos: {nodes_ab}")
print(f"Reduccion de nodos: {reduction:.2f}%")

if move_mm == move_ab:
    print("Ambos algoritmos coinciden en la decision (esperado).")
else:
    print("Nota: hubo diferencia en la decision; revise orden de exploracion y heuristica.")

Tablero de prueba (estado compartido):

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   | O |   |   |   |   | *
* |   | O | O | X | X |   |   | *
* |   | O | O | X | X | X |   | *
*********************************

Minimax puro -> mejor mov: 2, score: 1000, nodos: 1743
Alfa-Beta    -> mejor mov: 2, score: 1000, nodos: 297
Reduccion de nodos: 82.96%
Ambos algoritmos coinciden en la decision (esperado).


### Task 2.3

Como Minimax se limita a profundidad `d`, no siempre llega a un estado terminal.  

Por eso se implementa `evaluate_board(state, ai_player)` (función de evaluación heurística), que aproxima que tan bueno es un estado intermedio.  

Criterios principales:  

- Priorizar las columnas centrales.  

- Recompensar ventanas de 4 con 2 o 3 fichas propias.  

- Penalizar amenazas del oponente (3 en linea con un espacio).

In [66]:
def evaluate_window_strategic(current_window_elements, current_window_coords, board_state, ai_player):
    """
    Función auxiliar para evaluar una ventana de 4 fichas en el tablero.
    """
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2
    score = 0

    num_ai = 0
    num_opp = 0
    num_empty_playable = 0 # Contará solo los espacios vacíos que pueden ser ocupados inmediatamente

    # Analizar los elementos de la ventana y su jugabilidad
    for i in range(4):
        element = current_window_elements[i]
        r_coord, c_coord = current_window_coords[i]

        if element == ai_player:
            num_ai += 1
        elif element == opponent:
            num_opp += 1
        elif element == EMPTY:
            # Verificar si este espacio vacío es jugable según las reglas de gravedad
            # Es jugable si está en la fila inferior o si la celda inmediatamente debajo está ocupada.
            is_playable = (r_coord == ROWS - 1) or (board_state[r_coord + 1][c_coord] != EMPTY)
            if is_playable:
                num_empty_playable += 1

    # Oportunidades ofensivas de la IA
    if num_ai == 4:
        score += 1000  # Un 4 en raya completo (victoria inmediata).
    elif num_ai == 3 and num_empty_playable >= 1:
        score += 100   # Fuerte amenaza de 4 en raya: 3 propias con al menos un espacio jugable.
    elif num_ai == 2 and num_empty_playable >= 2:
        score += 10    # Potencial de 4 en raya: 2 propias con al menos dos espacios jugables.

    # Amenazas defensivas del oponente
    if num_opp == 4:
        score -= 1000  # Victoria del oponente (derrota inmediata).
    elif num_opp == 3 and num_empty_playable >= 1:
        score -= 90    # Amenaza crítica del oponente: 3 de él con al menos un espacio jugable.
    elif num_opp == 2 and num_empty_playable >= 2:
        score -= 5     # Potencial del oponente: 2 de él con al menos dos espacios jugables.

    return score

def evaluate_game_state(state, ai_player):
    """
    Función heurística para evaluar un estado de juego no terminal.
    Valora posiciones estrategicas para el jugador AI (ai_player),
    considerando la jugabilidad de los espacios vacíos.
    """
    score = 0
    b = state.board
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    # Estrategia 1: Prioridad a la columna central
    # La columna central es a menudo la mas valiosa en Connect Four
    # porque participa en mas lineas de 4 (horizontal, vertical, diagonales).
    center_col = COLS // 2
    # Contamos las fichas propias en la columna central y les damos un peso mayor.
    score += sum(1 for r in range(ROWS) if b[r][center_col] == ai_player) * 5

    # Estrategia 2: Evaluar todas las ventanas de 4 fichas (horizontal, vertical, diagonal)
    # y asignar puntuaciones basadas en el potencial de ganar o perder, considerando la gravedad.

    # Evaluar horizontales
    for r in range(ROWS):
        for c in range(COLS - 3):
            window_elements = [b[r][c+i] for i in range(4)]
            window_coords = [(r, c+i) for i in range(4)]
            score += evaluate_window_strategic(window_elements, window_coords, b, ai_player)

    # Evaluar verticales
    for c in range(COLS):
        for r in range(ROWS - 3):
            window_elements = [b[r+i][c] for i in range(4)]
            window_coords = [(r+i, c) for i in range(4)]
            score += evaluate_window_strategic(window_elements, window_coords, b, ai_player)

    # Evaluar diagonales descendentes (\)
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            window_elements = [b[r+i][c+i] for i in range(4)]
            window_coords = [(r+i, c+i) for i in range(4)]
            score += evaluate_window_strategic(window_elements, window_coords, b, ai_player)

    # Evaluar diagonales ascendentes (/)
    for r in range(3, ROWS): # Las diagonales ascendentes empiezan desde la fila 3 (índice 3)
        for c in range(COLS - 3):
            window_elements = [b[r-i][c+i] for i in range(4)]
            window_coords = [(r-i, c+i) for i in range(4)]
            score += evaluate_window_strategic(window_elements, window_coords, b, ai_player)

    # Estrategia 3: Deteccion temprana de un posible ganador o perdedor
    # Aunque Minimax ya tiene check_winner, este refuerza la evaluacion
    # en caso de que la profundidad 0 caiga justo en un estado donde
    # una victoria/derrota esta presente, pero no fue capturada por evaluate_window_strategic
    # (e.g., debido a la interaccion de multiples ventanas).
    if state.check_winner(ai_player):
        score += 10000 # Un valor muy alto para una victoria directa
    elif state.check_winner(opponent):
        score -= 10000 # Un valor muy bajo para una derrota directa

    return score

#### 1. Rediseño de Poda Alfa-Beta

Actualizamos la poda alfa-beta para que ahora utilice la heurística para calcular el próximo mejor movimiento.

In [67]:
def alphabeta_count_nodes(state, depth, alpha, beta, maximizing_player, ai_player, counter):
    counter["nodes"] += 1
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if state.is_terminal():
        winner = state.winner()

        if winner == ai_player:
            return None, 1000

        if winner == opponent:
            return None, -1000

        return None, 0

    if depth == 0:
        # Usar la nueva función heurística estratégica
        return None, evaluate_game_state(state, ai_player)

    valid_moves = state.actions()

    if maximizing_player:
        best_score = float("-inf")
        best_move = valid_moves[0] if valid_moves else None

        for col in valid_moves:
            child = state.succ(col, ai_player)
            _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, False, ai_player, counter)

            if score > best_score:
                best_score = score
                best_move = col

            # Actualiza alpha en nodo MAX.
            alpha = max(alpha, best_score)

            # Poda: MIN ya tiene una opcion mejor o igual en ancestros.
            if alpha >= beta:
                break

        return best_move, best_score

    best_score = float("inf")
    best_move = valid_moves[0] if valid_moves else None

    for col in valid_moves:
        child = state.succ(col, opponent)
        _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, True, ai_player, counter)

        if score < best_score:
            best_score = score
            best_move = col

        # Actualiza beta en nodo MIN.
        beta = min(beta, best_score)

        # Poda: MAX ya tiene una opcion mejor o igual en ancestros.
        if alpha >= beta:
            break

    return best_move, best_score

#### 2. Simulación 1

Agente Alfa-Beta juega contra Agente Aleatorio (para demostrar que sabe ganar)

In [68]:
# Prueba 1: Agente Alfa-Beta vs Agente Aleatorio
# Este fragmento de código es similar al de la prueba con el Agente Minimax
def play_alphabeta_vs_random(depth=5, ai_player=PLAYER_1, seed=None):
    if seed is not None:
        random.seed(seed)
    game = Connect4()
    current_player = PLAYER_1

    print(f"\n[INICIO DEL JUEGO] Agente Alfa-Beta (profundidad {depth}) vs. Agente Aleatorio")
    game.print_board()

    while not game.is_terminal():
        if current_player == ai_player:
            print(f"Turno del Agente Alfa-Beta ({ai_player})... ")
            # Esta vez, usamos get_best_move_alphabeta para que la IA juegue
            col = get_best_move_alphabeta(game, depth=depth, ai_player=ai_player)
            if col is None: # Si no hay movimientos válidos (ej. tablero lleno)
                break
            game.make_move(col, current_player)
            print(f"Alfa-Beta juega en columna {col + 1}")
        else:
            print(f"Turno del Agente Aleatorio ({current_player})... ")
            valid_actions = game.actions()
            if not valid_actions:
                break # No hay movimientos válidos
            col = random.choice(valid_actions)
            game.make_move(col, current_player)
            print(f"Aleatorio juega en columna {col + 1}")

        game.print_board()
        current_player = PLAYER_1 if current_player == PLAYER_2 else PLAYER_2

    print("¡Fin del Juego!")
    w = game.winner()
    if w is None:
        return "Empate"
    else:
        return f"Gana jugador {w}"


# Ejecutar la prueba 1
print("\n--- Demostración: Agente Alfa-Beta (X) vs. Agente Aleatorio (O) ---")
resultado_ia_vs_random = play_alphabeta_vs_random(depth=6, ai_player=PLAYER_1, seed=42)
print("[FIN] Resultado Final:", resultado_ia_vs_random)


--- Demostración: Agente Alfa-Beta (X) vs. Agente Aleatorio (O) ---

[INICIO DEL JUEGO] Agente Alfa-Beta (profundidad 6) vs. Agente Aleatorio

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
*********************************

Turno del Agente Alfa-Beta (X)... 
Alfa-Beta juega en columna 4

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   | X |   |   |   | *
*********************************

Turno del Agente Aleatorio (O)... 
Aleatorio juega en columna 6

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   

#### 3. Simulación 2

Agente Alfa-Beta juega contra Humano (para intentar ganarle)

In [69]:
# Prueba 2: Agente Alfa-Beta vs. Jugador Humano
# Juega como el jugador 'O' contra la IA 'X'.
def play_alphabeta_vs_human(depth=5, ai_player=PLAYER_1):
    game = Connect4()
    current_player = PLAYER_1

    print(f"\n[INICIO DEL JUEGO] Agente Alfa-Beta (profundidad {depth}) vs. Humano")
    print(f"Eres el jugador '{PLAYER_2}'. La IA es el jugador '{ai_player}'.")
    game.print_board()

    while not game.is_terminal():
        if current_player == ai_player:
            print(f"Turno del Agente Alfa-Beta ({ai_player})... ")
            col = get_best_move_alphabeta(game, depth=depth, ai_player=ai_player)
            if col is None:
                break
            game.make_move(col, current_player)
            print(f"Alfa-Beta juega en columna {col + 1}")
        else:
            print(f"Tu turno ({current_player}). Columnas disponibles: { [c + 1 for c in game.actions()] }")
            valid_move = False
            player_col = -1
            while not valid_move:
                try:
                    player_col = int(input("Elige una columna (1-7): ")) - 1 # Restar 1 para el índice de lista
                    if player_col in game.actions():
                        valid_move = True
                    else:
                        print("Columna inválida o llena. Intenta de nuevo.")
                except ValueError:
                    print("Entrada inválida. Por favor, ingresa un número.")

            game.make_move(player_col, current_player)
            print(f"Tú juegas en columna {col + 1}")

        game.print_board()
        current_player = PLAYER_1 if current_player == PLAYER_2 else PLAYER_2

    w = game.winner()
    print("¡Fin del Juego!")
    if w is None:
        return "Empate"
    else:
        return f"Gana jugador {w}"


# Ejecutar la prueba 2
# Para que la IA juegue como 'O', cambiar ai_player=PLAYER_2.
print("\n--- Demostración: IA Alfa-Beta (X) vs. Jugador Humano (O) ---")
resultado_ia_vs_humano = play_alphabeta_vs_human(depth=6, ai_player=PLAYER_1)
print("[FIN] Resultado Final:", resultado_ia_vs_humano)


--- Demostración: IA Alfa-Beta (X) vs. Jugador Humano (O) ---

[INICIO DEL JUEGO] Agente Alfa-Beta (profundidad 6) vs. Humano
Eres el jugador 'O'. La IA es el jugador 'X'.

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
*********************************

Turno del Agente Alfa-Beta (X)... 
Alfa-Beta juega en columna 4

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   | X |   |   |   | *
*********************************

Tu turno (O). Columnas disponibles: [1, 2, 3, 4, 5, 6, 7]
Elige una columna (1-7): 3
Tú juegas en columna 4

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* | 